In [3]:
import json

INPUT_PATH  = "crt-10k/cjpe_train_rr_segmented.jsonl"
OUTPUT_PATH = "crt-10k/cjpe_fact_arg_extracted_GPU.jsonl"

# ── Field mapping (original name → output name) ───────────────────────────────
FIELD_MAP = {
    "FAC"            : "FAC",
    "ARG_PETITIONER" : "ARG_P",
    "ARG_RESPONDENT" : "ARG_R",
}

extracted  = 0
skipped    = 0
empty_fac  = 0
empty_argp = 0
empty_argr = 0

with open(INPUT_PATH, "r", encoding="utf-8") as fin, \
     open(OUTPUT_PATH, "w", encoding="utf-8") as fout:

    for line in fin:
        line = line.strip()
        if not line:
            continue

        rec = json.loads(line)

        out = {
            "id"    : rec.get("id", ""),
            "FAC"   : rec.get("FAC",            "").strip(),
            "ARG_P" : rec.get("ARG_PETITIONER", "").strip(),
            "ARG_R" : rec.get("ARG_RESPONDENT", "").strip(),
            "label" : rec.get("label", -1),
        }

        # ── track which fields are empty ──────────────────────────────────────
        if not out["FAC"]:   empty_fac  += 1
        if not out["ARG_P"]: empty_argp += 1
        if not out["ARG_R"]: empty_argr += 1

        fout.write(json.dumps(out, ensure_ascii=False) + "\n")
        extracted += 1

# ═══════════════════════════════════════════════════════════════════════════════
# Summary
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 55)
print("EXTRACTION COMPLETE")
print("=" * 55)
print(f"  Input          : {INPUT_PATH}")
print(f"  Output         : {OUTPUT_PATH}")
print(f"  Total records  : {extracted:,}")
print()
print(f"  Field coverage (docs that HAVE the field):")
print(f"    FAC   : {extracted - empty_fac:>6,}  /  {extracted:,}  "
      f"({(extracted-empty_fac)/extracted*100:.1f}%)")
print(f"    ARG_P : {extracted - empty_argp:>6,}  /  {extracted:,}  "
      f"({(extracted-empty_argp)/extracted*100:.1f}%)")
print(f"    ARG_R : {extracted - empty_argr:>6,}  /  {extracted:,}  "
      f"({(extracted-empty_argr)/extracted*100:.1f}%)")

# ── Label distribution ────────────────────────────────────────────────────────
from collections import Counter
label_counts = Counter()
with open(OUTPUT_PATH) as f:
    for line in f:
        label_counts[json.loads(line)["label"]] += 1

print()
print(f"  Label distribution:")
for lbl, cnt in sorted(label_counts.items()):
    name = "ACCEPTED" if lbl == 1 else "REJECTED"
    print(f"    Label {lbl} ({name}) : {cnt:>6,}  ({cnt/extracted*100:.1f}%)")

# ── Sample output ─────────────────────────────────────────────────────────────
print()
print("  Sample output (first 3 records):")
with open(OUTPUT_PATH) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        rec = json.loads(line)
        print(f"\n  {'─'*52}")
        print(f"  ID    : {rec['id']}  |  Label: {rec['label']}")
        print(f"  FAC   : {str(rec['FAC'])[:120]}{'...' if len(str(rec['FAC']))>120 else ''}")
        print(f"  ARG_P : {str(rec['ARG_P'])[:120]}{'...' if len(str(rec['ARG_P']))>120 else ''}")
        print(f"  ARG_R : {str(rec['ARG_R'])[:120]}{'...' if len(str(rec['ARG_R']))>120 else ''}")

print(f"\n  Output saved to → {OUTPUT_PATH}")

EXTRACTION COMPLETE
  Input          : crt-10k/cjpe_train_rr_segmented.jsonl
  Output         : crt-10k/cjpe_fact_arg_extracted_GPU.jsonl
  Total records  : 32,191

  Field coverage (docs that HAVE the field):
    FAC   : 28,115  /  32,191  (87.3%)
    ARG_P : 18,319  /  32,191  (56.9%)
    ARG_R : 12,174  /  32,191  (37.8%)

  Label distribution:
    Label 0 (REJECTED) : 18,856  (58.6%)
    Label 1 (ACCEPTED) : 13,335  (41.4%)

  Sample output (first 3 records):

  ────────────────────────────────────────────────────
  ID    : 2020_1  |  Label: 0
  FAC   : 
  ARG_P : Some of the salient features of the Bill were paraphrased in the majority opinion delivered by S.R.
  ARG_R : 

  ────────────────────────────────────────────────────
  ID    : 2020_2  |  Label: 0
  FAC   : 
  ARG_P : PW-6 denied knowledge of what had happened to the lorry after its delivery to A1.
  ARG_R : 

  ────────────────────────────────────────────────────
  ID    : 2020_3  |  Label: 0
  FAC   : 
  ARG_P : The fun